# Task 3: Agentic Workflows · Multi-Agent Financial Research System

**Stack:** LangGraph · LangChain · OpenRouter (`openai/gpt-4o`, set via `OPENROUTER_MODEL`) · yfinance · ddgs · Pydantic v2

This notebook demonstrates every Task 3 requirement with visible output:

## 0. Setup

In [ ]:
# Clone and install when running in Colab; locally, run this notebook from task3_agentic/.
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Keirishan/CDAZZDEV-MLE-Keirishan.git"  

if IN_COLAB:
    repo_dir = pathlib.Path("/content/repo")
    if not repo_dir.exists():
        !git clone -q {REPO_URL} {repo_dir}
    os.chdir(repo_dir / "task3_agentic")
    !pip install -q -r requirements.txt

PROJECT_DIR = pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Project dir:", PROJECT_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 79.7 MB/s eta 0:00:00
Project dir: /content/repo/task3_agentic


In [ ]:
from agentic import get_settings, reset_settings
reset_settings()
s = get_settings()
print("Model:", s.model, "| sentiment model:", s.sentiment_model)
print("API key configured:", bool(s.openrouter_api_key))
print("Trace file:", s.trace_path)
print("Cache dir:", s.cache_dir)
assert s.openrouter_api_key, "Set OPENROUTER_API_KEY in Colab Secrets or .env, then re-run this cell."

Model: openai/gpt-4o | sentiment model: openai/gpt-4o
API key configured: True
Trace file: /content/repo/task3_agentic/logs/agent_trace.jsonl
Cache dir: /content/repo/task3_agentic/cache


In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display
from agentic import ResearchAgent, ResearchPipeline, to_markdown, tracer, console
from agentic.tools import (get_price_data, get_news, calculate_volatility, llm_sentiment,
                           web_search, failing)
from agentic.schemas import ToolResult
from agentic.memory import delete_cached

TICKER = s.default_ticker         
RESET_TRACE = True                
if RESET_TRACE:
    tracer.clear()
delete_cached(TICKER)              
print("Ticker:", TICKER)

Ticker: NVDA


## 1. The five tools

Each tool is a LangChain tool with a Pydantic argument schema. It returns a `ToolResult`
envelope `{ok, tool, data, error, source}` and **never raises**. Failures come back as
`ok=False` with a reason the agent can act on. Every call is traced (the `ok …` lines).

In [ ]:
results = {
    "get_price_data": get_price_data.invoke({"ticker": TICKER, "period": "1y"}),
    "calculate_volatility": calculate_volatility.invoke({"ticker": TICKER, "window": 30}),
    "get_news": get_news.invoke({"ticker": TICKER, "n": 10}),
    "web_search": web_search.invoke({"query": f"{TICKER} stock analyst outlook"}),
}
headlines = [i["title"] for i in (results["get_news"]["data"] or {}).get("items", [])][:5]
results["llm_sentiment"] = llm_sentiment.invoke({"headlines": headlines, "context": TICKER})

rows = []
for name, r in results.items():
    tr = ToolResult.model_validate(r)  
    data = tr.data if isinstance(tr.data, dict) else {}
    rows.append({"tool": name, "return type": type(r).__name__, "ok": tr.ok,
                 "data keys": ", ".join(list(data)[:8]), "error": tr.error})
pd.DataFrame(rows)

[SYSTEM        ]    ok get_price_data  1665 ms -> {"ticker": "NVDA", "as_of": "2026-10-06", "period": "1y", "rows_analysed": 253, "last_close": 239.24, "period_return_pct": 27.8186, "ytd_return_pct": 28.579, "high_52w": 243.37, "low_52w": 163.8954...
[SYSTEM        ]    ok calculate_volatility  5 ms -> {"ticker": "NVDA", "as_of": "2026-10-06", "window": 30, "annualised_vol": 0.3742, "vol_30d": 0.3742, "vol_60d": 0.371, "vol_90d": 0.3954, "vol_1y": 0.3759, "downside_vol": 0.2082, "max_drawdown": -...
[SYSTEM        ]    ok get_news  508 ms -> {"ticker": "NVDA", "count": 10, "items": [{"title": "Why NVDA, AMD, CRWD Stocks Surged To 52-Week Highs Today", "publisher": null, "published": "Wed, 07 Oct 2026 04:19:23 +0000", "url": "https://st...
[SYSTEM        ]    ok web_search  2320 ms -> {"query": "NVDA stock analyst outlook", "count": 6, "backend": "ddgs_text", "results": [{"title": "NVIDIA (NVDA) Stock Forecast & Analyst Price Targets", "snippet": "8 hours ago · Stock forecasts a...
[SYS

,tool,return type,ok,data keys,error
0,get_price_data,dict,True,"ticker, as_of, period, rows_analysed, last_clo...",None
1,calculate_volatility,dict,True,"ticker, as_of, window, annualised_vol, vol_30d...",None
2,get_news,dict,True,"ticker, count, items, sources_used",None
3,web_search,dict,True,"query, count, backend, results",None
4,llm_sentiment,dict,True,"count, overall_score, overall_label, counts, i...",None


In [ ]:
p = results["get_price_data"]["data"]
if p:
    print(json.dumps({k: p[k] for k in ["last_close", "sma50", "sma200", "rsi14", "macd", "macd_signal",
                                        "macd_hist", "bb_upper", "bb_lower", "momentum_signal", "flags"]}, indent=1))
v = results["calculate_volatility"]["data"]
if v:
    print(json.dumps(v, indent=1))

{
 "last_close": 239.24,
 "sma50": 219.6239,
 "sma200": 201.0306,
 "rsi14": 67.2719,
 "macd": 4.7349,
 "macd_signal": 3.3622,
 "macd_hist": 1.3727,
 "bb_upper": 240.2282,
 "bb_lower": 209.5238,
 "momentum_signal": "bullish",
 "flags": [
  "above_sma200",
  "sma50_above_sma200",
  "macd_above_signal",
  "macd_momentum_rising"
 ]
}
{
 "ticker": "NVDA",
 "as_of": "2026-10-06",
 "window": 30,
 "annualised_vol": 0.3742,
 "vol_30d": 0.3742,
 "vol_60d": 0.371,
 "vol_90d": 0.3954,
 "vol_1y": 0.3759,
 "downside_vol": 0.2082,
 "max_drawdown": -0.0832,
 "last_close": 239.24,
 "expected_move_90d_1sigma": 44.4536,
 "expected_move_90d_pct": 18.5812,
 "vol_regime": "normal"
}


## 2. Task 3A: single research agent (autonomous tool selection)

```mermaid
flowchart LR
  S([query]) --> A[agent: GPT-4o + 5 tools]
  A -- tool calls --> T[tool executor<br/>trace + allow-list]
  T -- observations --> A
  A -- enough evidence --> F[finalize: ResearchReport<br/>Pydantic validated]
  F --> E([report])
```

The system prompt lists goals, tools and rules, **never an order**. Each step below shows
the agent's `Thought` (what it learned and why it picks the next tool), the `CALL`, and the
observation.

In [7]:
agent = ResearchAgent()
res = agent.run(TICKER)
print("\nTool sequence chosen by the agent:", res.tool_sequence)


Task 3A research agent | ticker=NVDA | session=nvda-ce6f8e | run=bb3c44cb5842
[RESEARCH_AGENT] Query: Analyse the current financial health and market sentiment of NVDA. Identify the
top three risks to its share price over the next 90 days and suggest one
data-driven hedge strategy.
[RESEARCH_AGENT] -> CALL get_price_data(ticker="NVDA")
[RESEARCH_AGENT] -> CALL get_news(ticker="NVDA", n=10)
[RESEARCH_AGENT] -> CALL calculate_volatility(ticker="NVDA")
[RESEARCH_AGENT]    ok get_price_data  9 ms -> {"ticker": "NVDA", "as_of": "2026-10-06", "period": "1y", "rows_analysed": 253, "last_close": 239.24, "period_return_pct": 27.8186, "ytd_return_pct": 28.579, "high_52w": 243.37, "low_52w": 163.8954...
[RESEARCH_AGENT]    ok get_news  364 ms -> {"ticker": "NVDA", "count": 10, "items": [{"title": "Why NVDA, AMD, CRWD Stocks Surged To 52-Week Highs Today", "publisher": null, "published": "Wed, 07 Oct 2026 04:19:23 +0000", "url": "https://st...
[RESEARCH_AGENT]    ok calculate_volatility  7 ms -> 

In [8]:
display(Markdown(to_markdown(res.report)))

# Research brief: NVDA

## 1. Financial Health Summary

Nvidia is currently exhibiting strong financial health, characterized by a substantial market capitalization of approximately $5.78 trillion and robust profit margins of 63.66%. The company has demonstrated impressive revenue growth of 105.9% and maintains a low debt-to-equity ratio of 16.97, alongside a high current ratio of 4.59, indicating strong liquidity. The stock is trading near its 52-week high at $239.24, reflecting strong market confidence. Sentiment is generally positive, bolstered by significant demand for Nvidia's chips, notably from SpaceX's $40 billion purchase interest. However, the high P/E ratio suggests potential overvaluation concerns, and the stock's annualized volatility of 37.42% indicates potential for significant price swings.

## 2. Top Three Risks (next 90 days)

### 1. Valuation Concerns  ·  severity: medium

Nvidia's high trailing P/E ratio of 30.25 and forward P/E of 15.14 suggest that the stock may be overvalued, especially if the company's growth expectations are not met. This could lead to a correction in the stock price if investor sentiment shifts.

**Evidence**
- Trailing P/E ratio of 30.25 and forward P/E of 15.14 (source: get_price_data)
- Stock trading near 52-week high at $239.24 (source: get_price_data)

_Sources: get_price_data_

### 2. Market Volatility  ·  severity: high

The annualized volatility for Nvidia is 37.42%, with an expected 90-day move of ±18.58%. This high volatility indicates that the stock could experience significant price swings, which could be exacerbated by broader market conditions or company-specific news.

**Evidence**
- Annualized volatility of 37.42% (source: calculate_volatility)
- Expected 90-day move of ±18.58% (source: calculate_volatility)

_Sources: calculate_volatility_

### 3. Competitive Pressure  ·  severity: medium

Nvidia operates in a highly competitive tech sector. Any advancements by competitors could impact Nvidia's market position and affect its stock price. The company's ability to maintain its competitive edge is crucial for sustaining its current valuation.

**Evidence**
- Nvidia's market cap nearing $6 trillion, indicating high expectations (source: get_news)
- Comparison with other AI companies like Applied Materials (source: get_news)

_Sources: get_news_

## 3. Hedge Strategy Recommendation

**Protective Put**: Buy put options with a strike price close to $239.24, expiring in 90 days.

A protective put strategy is suitable given the high volatility and potential downside risks identified, such as valuation concerns and competitive pressures. This strategy will help mitigate potential losses if the stock price declines significantly.

**Sizing:** With an expected 90-day move of ±18.58%, put options should be purchased at a strike price near the current trading price of $239.24 to cover potential downside.

**Data used:** Expected 90-day move of ±18.58% (source: calculate_volatility); Current trading price of $239.24 (source: get_price_data)

---
_This report is generated automatically for an engineering assessment. It is not investment advice. Figures come from free public data sources that may be delayed or incomplete. Consult a licensed financial adviser before making investment decisions._

## 3. Task 3A: error handling with an injected tool failure

`get_news` is forced to fail (`failing("get_news")`). The agent observes `ok=false` and
must reach the same information another way (typically `web_search`), without stopping.

In [9]:
with failing("get_news"):
    res_fail = agent.run(TICKER)

seq = res_fail.tool_sequence
print("\nTool sequence:", seq)
for i, o in enumerate(res_fail.observations):
    if not o["result"]["ok"]:
        nxt = seq[i + 1] if i + 1 < len(seq) else "(finalize)"
        print(f"Observed failure of {o['tool']} -> next decision: {nxt}")
print("Report produced:", res_fail.report.generated_by, "| risks:", len(res_fail.report.top_risks))


Task 3A research agent | ticker=NVDA | session=nvda-abbce6 | run=71a483cae286
[RESEARCH_AGENT] Query: Analyse the current financial health and market sentiment of NVDA. Identify the
top three risks to its share price over the next 90 days and suggest one
data-driven hedge strategy.
[RESEARCH_AGENT] -> CALL get_price_data(ticker="NVDA")
[RESEARCH_AGENT] -> CALL get_news(ticker="NVDA", n=10)
[RESEARCH_AGENT] -> CALL calculate_volatility(ticker="NVDA")
[RESEARCH_AGENT]    ok get_price_data  17 ms -> {"ticker": "NVDA", "as_of": "2026-10-06", "period": "1y", "rows_analysed": 253, "last_close": 239.24, "period_return_pct": 27.8186, "ytd_return_pct": 28.579, "high_52w": 243.37, "low_52w": 163.8954...
[RESEARCH_AGENT]    INJECTED_FAILURE get_news  0 ms -> {"ok": false, "tool": "get_news", "data": null, "error": "Simulated outage (failure injection via FAIL_TOOLS). Try an alternative tool or approach.", "source": null}
[RESEARCH_AGENT]    ok calculate_volatility  7 ms -> {"ticker": "NVDA", "as

## 4. Task 3B: two-agent pipeline with critique loop

| Agent | Tools (enforced) | Output |
|---|---|---|
| A · Data Analyst | `get_price_data`, `calculate_volatility`, `llm_sentiment` | `DataBrief` (Pydantic) |
| B · Research Writer | `web_search`, `get_news` | `FinalReport` (Pydantic) |

Flow: `cache_check → analyst ⇄ tools → DataBrief → writer ⇄ tools → review → ClarificationRequest →
analyst (clarify) ⇄ tools → ClarificationResponse → review → FinalReport → save_cache`.
Agent A has no news access, so headline sentiment can only be produced by the two agents together:
B collects headlines and A scores them.

In [10]:
pipeline = ResearchPipeline()
out = pipeline.run(TICKER, force_refresh=True)
print(f"\ncache_hit={out.cache_hit} | tool calls={out.tool_calls} | clarification rounds={len(out.clarifications)}")


Task 3B pipeline | ticker=NVDA | session=pipeline-nvda-ba79e8 | run=cc0b0945425d
[SYSTEM        ] cache_check: force_refresh=True, ignoring NVDA_2026-10-07.json
[ANALYST       ] Thought: I will start by gathering the price data and key financial indicators for NVDA to understand its current financial health and market trends.
[ANALYST       ] -> CALL get_price_data(ticker="NVDA")
[ANALYST       ]    ok get_price_data  11 ms -> {"ticker": "NVDA", "as_of": "2026-10-06", "period": "1y", "rows_analysed": 253, "last_close": 239.24, "period_return_pct": 27.8186, "ytd_return_pct": 28.579, "high_52w": 243.37, "low_52w": 163.8954...
[ANALYST       ] LLM call failed (APIStatusError: Error code: 402 - {'error': {'message': 'This request would exceed your available credits given your current in-flight requests. Retry after in-flight requests settle, or add credits.', 'code': 402, 'metadata': {'reason': 'in_flight_budget_exhausted', 'limit_source': 'openrouter_in_flight_budget', '); continuing wit

In [11]:
display(Markdown(to_markdown(out.final_report)))

# Research brief: NVDA

> Built by the deterministic fallback because the LLM output failed validation.

## 1. Financial Health Summary

Automatic fallback summary (LLM report failed validation: APIStatusError: Error code: 402 - {'error': {'message': 'This request would exceed your available credits given your current in-flight requests. Retry after in-). Momentum signal: bullish; YTD return 28.6%; trailing P/E 30.2.

## 2. Top Three Risks (next 90 days)

### 1. Trend extension and momentum reversal  ·  severity: medium

The share price's position against its long-term averages and momentum indicators suggests the trend could reverse within 90 days.

**Evidence**
- Last close 239.24 vs SMA200 201.03
- RSI14 67.3, MACD histogram 1.373

_Sources: get_price_data_

### 2. Elevated volatility and drawdown risk  ·  severity: high

Realised volatility implies a wide range of outcomes over the next quarter.

**Evidence**
- Annualised volatility n/a (n/a-day window), regime: n/a
- Max drawdown in window n/a

_Sources: calculate_volatility_

### 3. News-driven sentiment shift  ·  severity: medium

Headline sentiment can move the price sharply around news and events.

**Evidence**
- Overall sentiment score n/a (n/a)
- Headline counts: n/a

_Sources: llm_sentiment_

## 3. Hedge Strategy Recommendation

**Protective put**: Buy ~90-day put options about one standard deviation below the current price.

Limits downside from the volatility and reversal risks while keeping upside.

**Sizing:** Volatility data unavailable; strike could not be computed.

**Data used:** last_close=239.24; expected_move_90d_1sigma=n/a

## Critique loop

Fallback report built from tool data.

## Sources

- get_price_data

---
_This report is generated automatically for an engineering assessment. It is not investment advice. Figures come from free public data sources that may be delayed or incomplete. Consult a licensed financial adviser before making investment decisions._

In [12]:
# The typed handoffs (validated Pydantic objects, not free text)
print("DataBrief type:", type(out.data_brief).__name__)
print("DataBrief fields:", list(out.data_brief.model_dump().keys()))
for i, c in enumerate(out.clarifications, 1):
    req = {k: v for k, v in c["request"].items() if k != "headlines"}
    print(f"\n--- Critique round {i}: ClarificationRequest (B -> A)")
    print(json.dumps(req, indent=1))
    print(f"(+ {len(c['request']['headlines'])} headlines attached for llm_sentiment)")
    print(f"--- ClarificationResponse (A -> B)")
    print(json.dumps({k: c['response'][k] for k in ('answer', 'tools_used')}, indent=1))
print("\nFinalReport.clarification_used:", out.final_report.clarification_used)
print("clarification_summary:", out.final_report.clarification_summary)

DataBrief type: DataBrief
DataBrief fields: ['ticker', 'as_of', 'price', 'volatility', 'sentiment', 'key_observations', 'quantitative_flags', 'data_gaps']

--- Critique round 1: ClarificationRequest (B -> A)
{
 "question": "Please score the sentiment of the headlines I collected, and give 90-day annualised volatility, downside volatility and maximum drawdown so I can size the hedge.",
 "requested_metrics": [
  "sentiment_overall_score",
  "vol_90d",
  "downside_vol_90d",
  "max_drawdown_90d"
 ],
 "reason": "Risk evidence needs sentiment; hedge sizing needs 90-day volatility."
}
(+ 0 headlines attached for llm_sentiment)
--- ClarificationResponse (A -> B)
{
 "answer": "No written answer could be produced (APIStatusError: Error code: 402 - {'error': {'message': 'This request would exceed your available credits given your current in-flight requests. Retry after in-flight requests settle, or add credits.', 'code': 402, 'metadata': {'reason': 'in_flight_budget_exhausted', 'limit_source': 'o

## 5. Tool restriction is enforced, not just prompted

The writer only *sees* its two tools, and the executor also rejects anything outside its
allow-list. Here we hand the writer's executor a fabricated `get_price_data` call.

In [13]:
from agentic.agents.writer import writer_tools_node
msgs, obs, executed = writer_tools_node.execute_calls(
    [{"name": "get_price_data", "args": {"ticker": TICKER}, "id": "demo-blocked"}])
print("Executed:", executed)
print("Message returned to the writer:", json.loads(msgs[0].content)["error"])
print("Trace record:", tracer.read()[-1])

# Tools each agent actually used in the section 4 pipeline run (from the trace file)
pipeline_calls = pd.DataFrame([r for r in tracer.read()
                               if r["kind"] == "tool" and r["run_id"] == out.run_id])
if pipeline_calls.empty:
    print("\nNo tool calls recorded for the section 4 run. Check its output for 'LLM call failed' and re-run section 4.")
else:
    display(pipeline_calls[["agent", "tool"]].value_counts().rename("calls").to_frame())

[WRITER        ]    BLOCKED get_price_data  0 ms -> Tool 'get_price_data' is not permitted for agent 'writer'. Allowed tools: ['get_news', 'web_search'].
Executed: 0
Message returned to the writer: Tool 'get_price_data' is not permitted for agent 'writer'. Allowed tools: ['get_news', 'web_search'].
Trace record: {'ts': '2026-10-07T04:56:31.370+00:00', 'run_id': '-', 'session': '-', 'agent': 'writer', 'kind': 'guard', 'tool': 'get_price_data', 'args': {'agent': 'writer'}, 'output': "Tool 'get_price_data' is not permitted for agent 'writer'. Allowed tools: ['get_news', 'web_search'].", 'duration_ms': 0.0, 'status': 'blocked'}


,,calls
agent,tool,
analyst,get_price_data,1


## 6. Task 3C: short-term memory (follow-up without re-fetching)

Same `thread_id` as the run in section 2. Tools are **still bound**; the agent answers from
the checkpointed conversation. Proof: the session's tool-call count and the number of `tool`
lines in `agent_trace.jsonl` are unchanged.

In [14]:
fu = agent.ask("What RSI(14) value and 30-day volatility did you retrieve earlier, and which tools did they come from?",
               session=res.session)
print("\nAnswer:", fu.answer)
print(f"\nSession tool calls: {fu.tool_calls_before} -> {fu.tool_calls_after}")
print(f"Trace tool lines for session: {fu.trace_tool_lines_before} -> {fu.trace_tool_lines_after}")
print("Answered from memory:", fu.answered_from_memory)


Task 3C follow-up | session=nvda-ce6f8e
[RESEARCH_AGENT] Question: What RSI(14) value and 30-day volatility did you retrieve earlier, and which tools did they come from?
[RESEARCH_AGENT] LLM call failed (APIStatusError: Error code: 402 - {'error': {'message': 'This request would exceed your available credits given your current in-flight requests. Retry after in-flight requests settle, or add credits.', 'code': 402, 'metadata': {'reason': 'in_flight_budget_exhausted', 'limit_source': 'openrouter_in_flight_budget', '); continuing without further tool calls.
[RESEARCH_AGENT] Tool calls in session before/after follow-up: 4 -> 4; trace tool lines: 4 -> 4

Answer: Thought: the language model call failed (APIStatusError: Error code: 402 - {'error': {'message': 'This request would exceed your available credits given your current in-flight requests. Retry after in-flight requests settle, or add credits.', 'code': 402, 'metadata': {'reason': 'in_flight_budget_exhausted', 'limit_source': 'openro

## 7. Task 3C: persistent cache

Section 4 saved `cache/{TICKER}_{YYYY-MM-DD}.json`. Running the pipeline again for the same
ticker on the same day goes `cache_check → load_cached_brief → END`: no agents, no tools.

In [15]:
second = pipeline.run(TICKER)
print(f"\ncache_hit={second.cache_hit} | tool calls this run={second.tool_calls} | "
      f"trace tool records for this run={tracer.count('tool', run_id=second.run_id)}")
print("Cache file:", second.cache_path)
print("Same report as the first run:", second.final_report == out.final_report)


Task 3B pipeline | ticker=NVDA | session=pipeline-nvda-07003b | run=2d759503e354
[SYSTEM        ]    HIT cache_lookup  1 ms -> loaded NVDA_2026-10-07.json
[SYSTEM        ] cache_check: HIT (NVDA_2026-10-07.json)
[SYSTEM        ]    HIT cache_lookup  0 ms -> loaded NVDA_2026-10-07.json
[SYSTEM        ] Loaded cached brief from 2026-10-07 (original run cc0b0945425d, 1 tool calls). No agents or tools run.

cache_hit=True | tool calls this run=0 | trace tool records for this run=0
Cache file: /content/repo/task3_agentic/cache/NVDA_2026-10-07.json
Same report as the first run: True


In [16]:
cached = json.loads(pathlib.Path(second.cache_path).read_text())
print(json.dumps({k: cached[k] for k in ("ticker", "date", "created_at", "model", "run_id", "tool_call_count")}, indent=1))

{
 "ticker": "NVDA",
 "date": "2026-10-07",
 "created_at": "2026-10-07T04:56:31.329467+00:00",
 "model": "openai/gpt-4o",
 "run_id": "cc0b0945425d",
 "tool_call_count": 1
}


## 8. Observability: `logs/agent_trace.jsonl`

One JSON line per tool call: `tool`, `args`, `output` (truncated to 200 characters),
`duration_ms`, plus `agent`, `run_id`, `session`, `status`. LLM calls, guard decisions,
validation failures and cache lookups are logged as extra `kind`s.

In [17]:
trace = pd.DataFrame(tracer.read())
print("Trace file:", tracer.path, "| records:", len(trace))
display(trace.groupby(["kind", "status"]).size().rename("records").to_frame())
tools_df = trace[trace.kind == "tool"]
display(tools_df.groupby(["agent", "tool"])["duration_ms"].agg(["count", "mean"]).round(1))
tools_df[["ts", "agent", "tool", "args", "output", "duration_ms", "status"]].head(12)

Trace file: /content/repo/task3_agentic/logs/agent_trace.jsonl | records: 41


records
kind  status                   
cache hit                     2
      saved                   1
guard blocked                 1
llm   exception              11
      ok                     11
tool  injected_failure        1
      ok                     14

count    mean
agent          tool                               
analyst        get_price_data            1    11.2
research_agent calculate_volatility      2     7.0
               get_news                  2   181.8
               get_price_data            2    13.0
               llm_sentiment             2  3172.6
               web_search                1  1043.7
system         calculate_volatility      1     5.4
               get_news                  1   507.8
               get_price_data            1  1665.4
               llm_sentiment             1  7710.9
               web_search                1  2319.5

,ts,agent,tool,args,output,duration_ms,status
0,2026-10-07T04:55:52.613+00:00,system,get_price_data,"{'ticker': 'NVDA', 'period': '1y'}","{""ticker"": ""NVDA"", ""as_of"": ""2026-10-06"", ""per...",1665.4,ok
1,2026-10-07T04:55:52.621+00:00,system,calculate_volatility,"{'ticker': 'NVDA', 'window': 30}","{""ticker"": ""NVDA"", ""as_of"": ""2026-10-06"", ""win...",5.4,ok
2,2026-10-07T04:55:53.131+00:00,system,get_news,"{'ticker': 'NVDA', 'n': 10}","{""ticker"": ""NVDA"", ""count"": 10, ""items"": [{""ti...",507.8,ok
3,2026-10-07T04:55:55.454+00:00,system,web_search,"{'query': 'NVDA stock analyst outlook', 'max_r...","{""query"": ""NVDA stock analyst outlook"", ""count...",2319.5,ok
5,2026-10-07T04:56:03.170+00:00,system,llm_sentiment,"{'headlines': ['Why NVDA, AMD, CRWD Stocks Sur...","{""count"": 5, ""overall_score"": 0.6667, ""overall...",7710.9,ok
7,2026-10-07T04:56:04.463+00:00,research_agent,get_price_data,"{'ticker': 'NVDA', 'period': '1y'}","{""ticker"": ""NVDA"", ""as_of"": ""2026-10-06"", ""per...",9.3,ok
8,2026-10-07T04:56:04.829+00:00,research_agent,get_news,"{'ticker': 'NVDA', 'n': 10}","{""ticker"": ""NVDA"", ""count"": 10, ""items"": [{""ti...",363.5,ok
9,2026-10-07T04:56:04.838+00:00,research_agent,calculate_volatility,"{'ticker': 'NVDA', 'window': 30}","{""ticker"": ""NVDA"", ""as_of"": ""2026-10-06"", ""win...",6.8,ok
12,2026-10-07T04:56:10.660+00:00,research_agent,llm_sentiment,"{'headlines': ['Why NVDA, AMD, CRWD Stocks Sur...","{""count"": 10, ""overall_score"": 0.5915, ""overal...",3783.0,ok
16,2026-10-07T04:56:21.114+00:00,research_agent,get_price_data,"{'ticker': 'NVDA', 'period': '1y'}","{""ticker"": ""NVDA"", ""as_of"": ""2026-10-06"", ""per...",16.8,ok


In [18]:
# First raw lines of the file, exactly as committed
with open(tracer.path) as fh:
    for _ in range(3):
        print(fh.readline().rstrip())

{"ts": "2026-10-07T04:55:52.613+00:00", "run_id": "-", "session": "-", "agent": "system", "kind": "tool", "tool": "get_price_data", "args": {"ticker": "NVDA", "period": "1y"}, "output": "{\"ticker\": \"NVDA\", \"as_of\": \"2026-10-06\", \"period\": \"1y\", \"rows_analysed\": 253, \"last_close\": 239.24, \"period_return_pct\": 27.8186, \"ytd_return_pct\": 28.579, \"high_52w\": 243.37, \"low_52w\": 163.8954...", "duration_ms": 1665.4, "status": "ok"}
{"ts": "2026-10-07T04:55:52.621+00:00", "run_id": "-", "session": "-", "agent": "system", "kind": "tool", "tool": "calculate_volatility", "args": {"ticker": "NVDA", "window": 30}, "output": "{\"ticker\": \"NVDA\", \"as_of\": \"2026-10-06\", \"window\": 30, \"annualised_vol\": 0.3742, \"vol_30d\": 0.3742, \"vol_60d\": 0.371, \"vol_90d\": 0.3954, \"vol_1y\": 0.3759, \"downside_vol\": 0.2082, \"max_drawdown\": -...", "duration_ms": 5.4, "status": "ok"}
{"ts": "2026-10-07T04:55:53.131+00:00", "run_id": "-", "session": "-", "agent": "system", "ki

### Bonus: observability platform

* **Streamlit dashboard:** `streamlit run dashboard.py` shows the trace file as a timeline,
  per-tool durations and failures.